# Creating Tables in SSMS: Data Types, Nullability, and Constraints

> **What this notebook teaches:** How to go from a designed Entity (from earlier normalization sessions) to an actual, physical table in SQL Server — choosing the right data type for every attribute, deciding NULL vs. NOT NULL for each, and applying the right constraints. This is **Physical Design**, the step right after Logical Design, and it's where every earlier theoretical decision (keys, relationships, normalization) finally becomes a real `CREATE TABLE` statement in SSMS.

> **All code in this notebook is real T-SQL**, meant to run in an SSMS Query Editor against a real SQL Server instance.

## 1. Where This Fits in the Overall Process

Recall the full pipeline:

```text
Business/Raw Data -> Entities -> Attributes -> Keys -> Relationships
    -> Normalization -> Logical Design -> Physical Design -> SQL Tables
```

This notebook is entirely about the last two steps, broken into a repeatable procedure:

```text
1. Pick the Entity                (e.g., Student)
2. List every Attribute            (every piece of info this entity needs)
3. For EACH attribute, decide:
       a. Data Type                (what kind of value, how much space)
       b. Nullable?                 (NULL or NOT NULL)
       c. Constraints               (PRIMARY KEY, UNIQUE, CHECK, DEFAULT, FOREIGN KEY)
4. Choose the Primary Key
5. Write CREATE TABLE
6. Verify with sample data
```

Steps 3a-3c are done **one attribute at a time** — this is the part that takes the most judgment, and it's where this notebook spends most of its time.

## 2. Choosing a Data Type — A Decision Guide

| The attribute is... | Use this type | Example |
|---|---|---|
| Short text, fixed maximum length | `NVARCHAR(n)` (Unicode) or `VARCHAR(n)` (English-only) | Name, city |
| Long free text, unpredictable length | `NVARCHAR(MAX)` | Notes, description |
| A fixed-length code (always exactly N characters) | `CHAR(n)` / `NCHAR(n)` | National ID (always 10 digits) |
| A whole number | `INT` (or `TINYINT`/`SMALLINT`/`BIGINT` depending on range) | Age, quantity |
| Money or precise decimals | `DECIMAL(p,s)` | Price, salary |
| A date only (no time) | `DATE` | Birth date, order date |
| A date and time | `DATETIME2` | Created timestamp |
| True/false | `BIT` | IsActive |
| Binary data (images, files) | `VARBINARY(MAX)` | Photo, document |

**The general rule:** don't default to `NVARCHAR(MAX)` for everything — a tighter, more specific type documents the attribute's real-world constraints directly in the schema, and helps SQL Server store and index it more efficiently.

**`CHAR` vs. `VARCHAR`:** use `CHAR(n)` only when the value is genuinely *always* exactly `n` characters (like a national code) — otherwise `VARCHAR`/`NVARCHAR` avoids wasting space padding shorter values.

## Persian (Jalali) Dates in SQL Server

SQL Server does not have a separate data type for the Persian (Jalali) calendar. The `DATE` data type stores dates using the Gregorian calendar.

For example:

```text
2026-09-21
```

represents a Gregorian date.

If users enter dates in the Persian calendar, such as:

```text
1405/06/30
```

the recommended approach is to **convert the Persian date to its Gregorian equivalent before storing it in the database**.

The general process is:

```text
Persian Date entered by the user
            ↓
     Convert Jalali → Gregorian
            ↓
       Store as DATE
            ↓
     2026-09-21
```

When the date needs to be displayed to the user, the application can convert the Gregorian date back to the Persian calendar:

```text
DATE stored in SQL Server
            ↓
     Convert Gregorian → Jalali
            ↓
       1405/06/30
```

### Why should we store it as `DATE`?

Storing dates as `DATE` allows SQL Server to correctly perform date-related operations, such as:

* Comparing dates
* Sorting dates
* Calculating the difference between dates
* Adding or subtracting days
* Filtering records by date

For example:

```sql
SELECT *
FROM Instructors
WHERE StartDate > '2025-01-01';
```

### What should we avoid?

A common but less desirable approach is storing a Persian date as a string:

```sql
StartDate VARCHAR(10)
```

For example:

```text
1405/06/30
1404/12/15
1405/01/20
```

In this case, SQL Server treats the value as text rather than as a date. This makes date calculations, comparisons, sorting, and filtering more difficult and can lead to errors.

### Recommended Approach

For a database that uses Persian dates in its user interface, a good design is:

```text
User Input
1405/06/30
     ↓
Jalali → Gregorian conversion
     ↓
SQL Server
DATE
2026-09-21
```

The database stores the date using the standard `DATE` data type, while the application is responsible for converting the date to the Persian calendar when displaying it to users.

> **Key point:** The calendar used for displaying a date to the user does not have to be the same as the format used to store the date in the database.


## Converting Between Persian and Gregorian Dates in SQL Server

SQL Server does not provide a built-in function such as `JalaliToGregorian()` for directly converting a Persian (Jalali) date to a Gregorian `DATE`.

However, SQL Server provides several functions and features that can be used for date formatting and conversion.

### 1. Converting a Gregorian Date to Persian for Display

SQL Server's `FORMAT()` function can use the `fa-IR` culture to display a Gregorian date using the Persian calendar.

```sql
SELECT FORMAT(
    CAST('2026-09-21' AS DATE),
    'yyyy/MM/dd',
    'fa-IR'
) AS PersianDate;
```

Example output:

```text
1405/06/30
```

Here:

* `CAST(... AS DATE)` converts the value to the SQL Server `DATE` data type.
* `'yyyy/MM/dd'` specifies the display format.
* `'fa-IR'` specifies the Persian (Iranian) culture.

> `FORMAT()` is mainly useful for displaying dates. It should not be used as the main method for storing Persian dates in the database.

---

### 2. Converting a Persian Date to Gregorian

SQL Server does **not** have a built-in function that directly converts a string such as:

```text
1405/06/30
```

into a Gregorian `DATE`.

For example, this should not be treated as a normal date conversion:

```sql
-- This does not directly convert a Jalali date to Gregorian.
SELECT CONVERT(DATE, '1405/06/30');
```

The value `1405/06/30` is a Persian calendar date, while SQL Server's `DATE` type uses the Gregorian calendar.

There are two common approaches.

### Approach 1: Convert in the Application

The application can convert the Persian date to Gregorian before sending it to SQL Server.

```text
User Input
    ↓
1405/06/30
    ↓
Jalali → Gregorian conversion
    ↓
2026-09-21
    ↓
SQL Server DATE
```

This is often the preferred approach because date conversion is handled by the application layer, while SQL Server stores a standard `DATE` value.

---

### Approach 2: Create a User-Defined Function

If the conversion must be performed inside SQL Server, we can create a **User-Defined Function (UDF)** that implements the Jalali-to-Gregorian conversion algorithm.

For example, the function could be used like this:

```sql
SELECT dbo.JalaliToGregorian('1405/06/30') AS GregorianDate;
```

The function would return:

```text
2026-09-21
```

A custom function is useful when Persian dates are already stored or received as strings and the conversion needs to happen inside SQL Server.

---

### 3. Displaying Stored Dates as Persian Dates

If the database stores:

```sql
StartDate DATE
```

with a value such as:

```text
2026-09-21
```

we can display it as a Persian date:

```sql
SELECT
    StartDate,
    FORMAT(StartDate, 'yyyy/MM/dd', 'fa-IR') AS PersianStartDate
FROM Instructors;
```

The database still stores:

```text
2026-09-21
```

while the query result can display:

```text
1405/06/30
```

This keeps the database structure clean while allowing users to work with the Persian calendar.

### Recommended Design

For most applications, the recommended architecture is:

```text
                 Application
                      │
       ┌──────────────┴──────────────┐
       │                             │
   User Input                   Display
  Persian Date                Persian Date
  1405/06/30                  1405/06/30
       │                             ▲
       ↓                             │
Jalali → Gregorian            Gregorian → Persian
       │                             │
       ↓                             │
     SQL Server  ────────────────→ FORMAT()
       │
       ↓
    DATE
2026-09-21
```

### Key Point

> **Store dates as `DATE` or `DATETIME2` in SQL Server and handle Persian-calendar conversion at the application or presentation layer whenever possible.**

This allows SQL Server to perform date comparisons, sorting, filtering, and date arithmetic correctly while still allowing users to see dates in the Persian calendar.


## 2b. String Data Types in Detail — The Full Classification

The "text" row in the table above hides a real decision tree. SQL Server's string types split cleanly into two dimensions: **Fixed vs. Variable length**, and **Non-Unicode vs. Unicode**.

```text
String Data Types
|
+-- Fixed Length
|     +-- CHAR      (non-Unicode)
|     +-- NCHAR     (Unicode)
|
+-- Variable Length
      +-- VARCHAR   (non-Unicode)
      +-- NVARCHAR  (Unicode)
```

**Reading the `N` prefix:** wherever a type starts with `N` (`NCHAR`, `NVARCHAR`), it means **Unicode** — and Unicode storage costs **2 bytes per character** instead of 1 (this is exactly why `FirstNameEn`/`LastNameEn` correctly avoid the `N` prefix — they never need the extra byte per character, since they're guaranteed English-only).

### Size Limits — Regular vs. Large Object ("Rich") Types

| Category | Types | Max Size |
|---|---|---|
| Regular fixed/variable | `CHAR`, `VARCHAR` | **8,000 bytes** (= 8,000 characters, since 1 byte each) |
| Regular fixed/variable, Unicode | `NCHAR`, `NVARCHAR` | **4,000 characters** (= 8,000 bytes, since 2 bytes each) |
| Large Object ("Rich" / MAX types) | `VARCHAR(MAX)`, `NVARCHAR(MAX)` | up to **2 GB** |
| Large Object, binary | `VARBINARY(MAX)` | up to **2 GB** |

Notice `NCHAR`/`NVARCHAR`'s character limit (4,000) is exactly half of `CHAR`/`VARCHAR`'s (8,000) — both cap out at the same 8,000-**byte** ceiling, just measured differently because Unicode characters cost twice as much space each.

### "Rich" Data Types — Large Object (LOB) Storage

Once a column might genuinely need more than 8,000 bytes — full documents, long articles, or actual files — the `MAX` variants take over:

```text
VARCHAR(MAX)    -> very long non-Unicode text, up to 2 GB
NVARCHAR(MAX)   -> very long Unicode text, up to 2 GB
VARBINARY(MAX)  -> BLOB (Binary Large Object) -- raw binary data, up to 2 GB
```

**`VARBINARY(MAX)` is exactly what `Picture` used in the Student table earlier** — a **BLOB** (Binary Large Object) is the general term for a column storing an entire file's raw bytes directly in the database: an image (`.jpg`), an audio file (`.mp3`), a document (`.pdf`), or any other binary file type. SQL Server doesn't care what kind of file it is — it's just bytes; the *meaning* of those bytes (that they form a valid JPEG, for instance) is entirely up to the application reading them back out.

In [ ]:
-- Demonstrating the size boundary directly
DECLARE @ShortText VARCHAR(8000) = REPLICATE('A', 8000);   -- fits, right at the limit
-- DECLARE @TooLong VARCHAR(8000) = REPLICATE('A', 8001);  -- would truncate/error -- exceeds 8000

DECLARE @LongText VARCHAR(MAX) = REPLICATE('A', 9000);      -- no problem at all for MAX types
SELECT LEN(@ShortText) AS ShortTextLength, LEN(@LongText) AS LongTextLength;

-- Storing an actual file as a BLOB (conceptual -- OPENROWSET needs a real file path in SSMS)
-- INSERT INTO Student (Picture)
-- SELECT BulkColumn
-- FROM OPENROWSET(BULK 'C:\Photos\ali.jpg', SINGLE_BLOB) AS ImageFile;

## 2c. Numeric Data Types in Detail -- Exact Numbers

Just like strings split into Fixed/Variable, numeric types split by **how many bits they reserve**, which directly determines their range.

| Data Type | Storage | Bits | Range | Max Digits |
|---|---|---|---|---|
| `BIT` | ~1 bit (packed) | 1 | `0` or `1` (True / False) | 1 |
| `TINYINT` | 1 byte | 8 | `0` to `255` | 3 |
| `SMALLINT` | 2 bytes | 16 | `-32,768` to `32,767` | 5 |
| `INT` | 4 bytes | 32 | `-2,147,483,648` to `2,147,483,647` | 10 |
| `BIGINT` | 8 bytes | 64 | `-9,223,372,036,854,775,808` to `9,223,372,036,854,775,807` | 19 |

**Why these ranges specifically:**

```text
TINYINT:  8 bits  -> 2^8  = 256 values   -> 0 to 255 (unsigned -- no sign bit used)
SMALLINT: 16 bits -> 2^16 = 65,536 values -> split between negative and positive
INT:      32 bits -> 2^32 = ~4.3 billion  -> split between negative and positive
BIGINT:   64 bits -> 2^64 = ~18.4 quintillion -> split between negative and positive
```

`TINYINT` is the one exception -- it has no negative range at all, so its full 256 combinations go entirely to `0`-`255`.

### The Distinction That Matters Most: Digit Count Is Not Storage Size

```text
WRONG:  INT -> 10 digits -> "must need 10 bytes"
RIGHT:  INT -> a FIXED 4 bytes, regardless of value -- which happens to top out at ~2.1 billion (10 digits)
```

`INT` costs exactly **4 bytes** whether the stored value is `5` or `2,147,483,647` -- storage size depends on the *declared type*, never on the actual value inside it.

### Why Getting the Exact Size Right Actually Matters

This isn't pedantic -- it has three concrete, measurable consequences, especially in high-volume systems like banking.

**1. Storage cost, multiplied by row count.** A banking system with 50 million transaction rows, using `BIGINT` (8 bytes) where `INT` (4 bytes) would have sufficed:

```text
Extra cost per row: 4 bytes
Total extra cost:   4 bytes x 50,000,000 rows ~= 190 MB -- for ONE column, on ONE table
```

Real systems have dozens of such columns across dozens of tables -- this waste compounds at scale even though it's invisible on a small practice table.

**2. Performance -- rows per page.** SQL Server stores data in fixed 8 KB pages (Session 10). Smaller rows mean more rows fit per page, meaning fewer pages need to be read for the same query -- a few wasted bytes per row can mean 10%+ more disk I/O across a large table's every scan, backup, and index rebuild.

**3. Correctness -- overflow risk (the opposite mistake).** Choosing too small a type doesn't fail quietly -- it fails on `INSERT`, in production, the moment the range is exceeded:

```text
Msg 220, Level 16, State 2:
Arithmetic overflow error for data type int, value = 2147483648.
```

In a live banking system, that's an outage on exactly the table that can least afford one -- not a bug caught safely in testing.

**The practical rule:** pick the smallest type that comfortably covers the column's *realistic maximum*, with headroom -- not the default, and not blind over-provisioning with `BIGINT` everywhere either.

In [ ]:
-- Demonstrating storage difference directly
SELECT
    DATALENGTH(CAST(100 AS TINYINT))   AS TinyInt_Bytes,
    DATALENGTH(CAST(100 AS SMALLINT))  AS SmallInt_Bytes,
    DATALENGTH(CAST(100 AS INT))       AS Int_Bytes,
    DATALENGTH(CAST(100 AS BIGINT))    AS BigInt_Bytes;
-- Expected: 1, 2, 4, 8 -- SAME value (100), different storage cost per declared type

-- Demonstrating overflow failure directly
DECLARE @Overflow INT = 2147483647;
-- SET @Overflow = @Overflow + 1;   -- uncomment to see: Arithmetic overflow error

## 3. NULL vs. NOT NULL — The Decision Rule

Ask one question per attribute:

> **"Is this value guaranteed to exist for every single record, at the moment the record is first created?"**

- **Yes -> `NOT NULL`** — e.g., a student's first name; you can't have a student record with no name at all.
- **No -> `NULL` allowed** — e.g., a student's job; many students don't have one yet, and that's a legitimate, meaningful absence of data (not a data quality error, as covered in Session 1's Missing Data concept).

**Common mistake to avoid:** marking everything `NOT NULL` "to be safe" — this just forces placeholder values (empty strings, fake dates) into the database when data is genuinely unknown, which is worse than an honest `NULL`.

## 4. Constraint Types — Quick Reference

| Constraint | Purpose | Example |
|---|---|---|
| `PRIMARY KEY` | Uniquely identifies each row | `StudentID INT PRIMARY KEY` |
| `FOREIGN KEY` | Enforces a relationship to another table | `MajorCode INT REFERENCES Major(MajorCode)` |
| `UNIQUE` | No two rows can share this value, but it's not the Primary Key | `NationalCode CHAR(10) UNIQUE` |
| `NOT NULL` | Value can never be missing | covered above |
| `CHECK` | Restricts values to a valid range/pattern | `CHECK (Gender IN ('M','F'))` |
| `DEFAULT` | Auto-fills a value when none is given | `DEFAULT GETDATE()` |

Every one of these is something you decide **per attribute**, during the same pass where you're choosing data type and nullability — they're not a separate, later step.

## 4a. UNIQUE Constraint -- Defining It After the Table Already Exists

`UNIQUE` is a separate constraint from `PRIMARY KEY`, and it's common to add it *after* a table is created, once you realize a non-key column also needs to guarantee no duplicates.

### UNIQUE vs. PRIMARY KEY

| | `PRIMARY KEY` | `UNIQUE` |
|---|---|---|
| How many per table | Exactly one | As many as needed |
| Allows `NULL`? | Never | Allows **one** `NULL` (SQL Server specifically) |
| Typical use | The main identifier | A secondary "must not repeat" rule (email, national code, student number) |

A table has only one `PRIMARY KEY`, but can have several `UNIQUE` constraints -- exactly the `Student` table's situation later in this notebook: `StudentID` is the Primary Key, while `StudentNumber` and `NationalCode` each need their own independent uniqueness rule.

In [ ]:
-- Adding a UNIQUE constraint AFTER the table already exists
ALTER TABLE Student
ADD CONSTRAINT UQ_Student_StudentNumber UNIQUE (StudentNumber);

ALTER TABLE Student
ADD CONSTRAINT UQ_Student_NationalCode UNIQUE (NationalCode);

**Why name the constraint explicitly (`UQ_Student_StudentNumber`) instead of letting SQL Server auto-generate a name?** An explicit name makes future maintenance far easier -- dropping it later is a clean, readable statement:

```sql
ALTER TABLE Student
DROP CONSTRAINT UQ_Student_StudentNumber;
```

And when a duplicate value is rejected, the error message references the constraint by name -- a name like `UQ_Student_StudentNumber` immediately tells you *which* rule was violated, where an auto-generated name (`UQ__Student__A1B2C3D4`) tells you nothing useful at a glance.



## 4b. Constraints as the Enforcement of 1NF

Two of the settings covered above aren't just implementation details — they're how SQL Server **physically enforces** two of Normalization's earliest rules.

Recall 1NF's three rules:

```text
1. Every row must be unique
2. Every value in a column must be the same data type   <- enforced by Data Type
3. Every field must hold a single, atomic value          <- reinforced by NOT NULL
```

**Data Type is a constraint, not just a description.** Writing `Age INT` isn't a hint about what *should* go in that column — it's a hard rule the engine enforces on every `INSERT`. Try to insert the text `'twenty five'` into an `INT` column and SQL Server rejects it outright. This is exactly 1NF's rule #2, made physically unbreakable rather than just a design guideline someone has to remember to follow.

**`NOT NULL` complements rule #3.** 1NF requires every field to hold one real, atomic value — `NOT NULL` is what guarantees that value is actually *there* at all, rather than silently absent. Data Type constrains *what kind* of value is allowed; `NOT NULL` constrains *whether a value exists*. Together, they're the column-level machinery that makes 1NF something the database enforces automatically, not something you just hope was followed during design.

### Applied Example: Why `InsuranceCode` Should Be `VARCHAR`, Not `INT`

A good real-world test of "is this a data-type-constraint decision": is the column a **number to calculate with**, or an **identifier that happens to look like digits**?

```text
InsuranceCode INT           <- WRONG: loses leading zeros (0123456789 -> 123456789),
                                can't hold a future format change (letters, dashes)
InsuranceCode VARCHAR(20)   <- RIGHT: preserves the code exactly as issued, and
                                nobody will ever run SUM() or AVG() on an insurance code
```

The same reasoning already applied to `NationalCode CHAR(10)` earlier in this notebook: if something is fundamentally an **identifier**, not a **quantity**, its data type should say so — a numeric type on an identifier column is itself a subtle Data-Type-Constraint mistake, even though the values happen to look like numbers.

---
# 5. Full Worked Example — Sematec Student Table

**Entity:** `Student` (People category, from Session 9's five-category framework)

Following the naming convention from Session 10 (**PascalCase**), and the bilingual requirement — English and Persian versions of the name, using `Persian_100_CI_AS` (Session 11) for the Persian columns.

## Attribute-by-Attribute Design Decisions

| # | Attribute (Persian) | Column Name | Data Type | Nullable? | Constraint | Reasoning |
|---|---|---|---|---|---|---|
| 1 | -- | `StudentID` | `INT IDENTITY(1,1)` | NOT NULL | `PRIMARY KEY` | Surrogate Key (Session 8) -- small, stable, auto-generated |
| 2 | شماره دانشجویی | `StudentNumber` | `VARCHAR(15)` | NOT NULL | `UNIQUE` | Business Key -- the official student number; must be unique but isn't the ideal PK (could theoretically be reformatted) |
| 3 | کد ملی | `NationalCode` | `CHAR(10)` | NOT NULL | `UNIQUE`, `CHECK (LEN(NationalCode) = 10)` | Iranian national codes are always exactly 10 digits -- fixed-length `CHAR` fits, and must be unique per person |
| 4 | نام (انگلیسی) | `FirstNameEn` | `VARCHAR(50)` | NOT NULL | -- | English-only content -> no Unicode needed, saves storage vs NVARCHAR |
| 5 | نام خانوادگی (انگلیسی) | `LastNameEn` | `VARCHAR(50)` | NOT NULL | -- | Same reasoning |
| 6 | نام (فارسی) | `FirstNameFa` | `NVARCHAR(50) COLLATE Persian_100_CI_AS` | NOT NULL | -- | Persian collation for correct sort order  |
| 7 | نام خانوادگی (فارسی) | `LastNameFa` | `NVARCHAR(50) COLLATE Persian_100_CI_AS` | NOT NULL | -- | Same reasoning |
| 8 | جنسیت | `Gender` | `NVarCHAR(10)` | NOT NULL | `CHECK (Gender IN ('M','F'))` استاد میگه این مدلی اگه تعریف کنیم مطلوب نیست نباید جوری تعریف کنیم که تفسیر شدنی باشه باید کامل وارد بشه | Small, fixed domain -- a CHECK constraint is simpler than a lookup table for just two values |
| 9 | تاریخ تولد | `BirthDate` | `DATE` | NOT NULL | `CHECK (BirthDate <= GETDATE())` | Date only, no time needed; the CHECK prevents an impossible future birth date (Session on Outliers vs. Noise -- this is exactly a Noise-prevention rule) |
| 10 | تحصیلات | `Education` | `NVARCHAR(100)` | NULL | -- | Not always known or applicable at registration time |
| 11 | شغل | `Job` | `NVARCHAR(100)` | NULL | -- | Many students don't have a job -- a legitimate, meaningful NULL |
| 12 | رشته تحصیلی | `FieldOfStudy` | `NVARCHAR(100)` | NOT NULL | -- | Core to why the student is enrolled -- should always be known |
| 13 | آدرس | `Address` | `NVARCHAR(500)` | NULL | -- | Long free text; optional at registration |
| 14 | شماره تماس | `PhoneNumber` | `VARCHAR(15)` | NULL | -- | Optional; kept as VARCHAR since phone numbers aren't used in arithmetic and may include a leading `0` or `+` |
| 15 | تصویر | `Picture` | `VARBINARY(MAX)` | NULL | -- | Stores the actual photo file as binary data; optional |

### Why Both `FirstNameEn`/`FirstNameFa` Instead of One Column?

This is a direct application of the "can there be more than one value per entity instance" question from Session 9 -- here it's not about repetition (1-to-many), but about needing **two parallel versions of the same fact** (an English spelling and a Persian spelling), which is exactly why they're two separate columns rather than one column holding mixed-language data.
### VARCHAR vs. NVARCHAR -- Applied Per Column, Not Per Table

A table can (and often should) mix both: `FirstNameEn`/`LastNameEn` are guaranteed to contain only English characters, so they use plain `VARCHAR` -- no Unicode overhead needed. `FirstNameFa`/`LastNameFa`, `Education`, `Job`, `FieldOfStudy`, and `Address` are expected to contain Persian text, so they use `NVARCHAR` with the Persian collation from Session 11. `NationalCode`, `StudentNumber`, `PhoneNumber`, and `Gender` are purely numeric/coded values with no linguistic content at all, so they use `VARCHAR`/`CHAR` regardless of language.

**The deciding question for every column is simply: "Will this column ever need to store a non-Latin character?"** If no, `VARCHAR`/`CHAR` is both correct and more storage-efficient (roughly half the space of the Unicode equivalent) -- using `NVARCHAR` "just in case" for a column that's guaranteed English-only is a common but avoidable inefficiency.

## 6. The Full `CREATE TABLE` Statement

In [ ]:
USE SematecLMS;

CREATE TABLE Student (
    StudentID      INT IDENTITY(1,1) PRIMARY KEY,
    StudentNumber  VARCHAR(15)  NOT NULL UNIQUE,
    NationalCode   CHAR(10)     NOT NULL UNIQUE
                   CHECK (LEN(NationalCode) = 10),
    FirstNameEn    VARCHAR(50)  NOT NULL,
    LastNameEn     VARCHAR(50)  NOT NULL,
    FirstNameFa    NVARCHAR(50) COLLATE Persian_100_CI_AS NOT NULL,
    LastNameFa     NVARCHAR(50) COLLATE Persian_100_CI_AS NOT NULL,
    Gender         CHAR(1)      NOT NULL
                   CHECK (Gender IN ('M','F')),
    BirthDate      DATE         NOT NULL
                   CHECK (BirthDate <= GETDATE()),
    Education      NVARCHAR(100) NULL,
    Job            NVARCHAR(100) NULL,
    FieldOfStudy   NVARCHAR(100) NOT NULL,
    Address        NVARCHAR(500) NULL,
    PhoneNumber    VARCHAR(15)   NULL,
    Picture        VARBINARY(MAX) NULL
);

## 7. Creating This Same Table via the SSMS GUI (Equivalent Path)

Recall : every table can be built either by writing T-SQL directly (above) or through the graphical designer. For this table, the GUI path would be:

1. Object Explorer -> expand `SematecLMS` -> right-click **Tables** -> **New Table...**
2. For each row in the design grid: enter the **Column Name**, pick the **Data Type** from the dropdown, and check/uncheck **Allow Nulls**
3. Right-click the `StudentID` row -> **Set Primary Key**
4. For `CHECK` and `UNIQUE` constraints not directly exposed in the grid: right-click anywhere in the designer -> **Indexes/Keys** (for `UNIQUE`) or **Check Constraints** (for `CHECK`), and add them there
5. Save the table (Ctrl+S), which prompts for a table name if not already set

Both paths produce an identical result — the GUI is friendlier while learning, but the T-SQL above is what actually gets generated and executed underneath it either way.

## 8. Verifying the Design With Sample Data

In [ ]:
INSERT INTO Student
    (StudentNumber, NationalCode, FirstNameEn, LastNameEn, FirstNameFa, LastNameFa,
     Gender, BirthDate, Education, Job, FieldOfStudy, Address, PhoneNumber)
VALUES
    ('99012345', '0012345678', 'Ali', 'Rahmani', N'علی', N'رحمانی',
     'M', '2001-03-15', N'دیپلم', NULL, N'مهندسی کامپیوتر',
     N'تهران', '09121234567');

SELECT StudentNumber, FirstNameEn, LastNameEn, FirstNameFa, LastNameFa, Gender, BirthDate, FieldOfStudy
FROM Student;

### Confirming the Constraints Actually Work

A schema is only as good as the constraints that enforce it — worth testing that invalid data genuinely gets rejected:

In [ ]:
-- This should FAIL -- Gender must be 'M' or 'F' only
INSERT INTO Student
    (StudentNumber, NationalCode, FirstNameEn, LastNameEn, FirstNameFa, LastNameFa,
     Gender, BirthDate, FieldOfStudy)
VALUES
    ('99012346', '0012345679', 'Sara', 'Karimi', N'سارا', N'کریمی',
     'X', '2001-05-20', N'ریاضی');
-- Expected: CHECK constraint violation on Gender

-- This should also FAIL -- NationalCode must be exactly 10 characters
INSERT INTO Student
    (StudentNumber, NationalCode, FirstNameEn, LastNameEn, FirstNameFa, LastNameFa,
     Gender, BirthDate, FieldOfStudy)
VALUES
    ('99012347', '12345', 'Reza', 'Shaygan', N'رضا', N'شایگان',
     'M', '2000-01-01', N'برق');
-- Expected: CHECK constraint violation on NationalCode length

## 9. A Note on `Education`, `Job`, and `FieldOfStudy` — Free Text vs. Normalized Lookup Tables

This table stores these three as plain `NVARCHAR` free text, which is the simplest starting point. But applying Session 6-9's normalization thinking, there's a legitimate design question here: **should `FieldOfStudy` instead be its own table**, like `ProductCategory` was for `Product`?

```text
Student.FieldOfStudy (free text)          vs.          Student.FieldOfStudyCode -> FieldOfStudy table
"Computer Engineering"                                   101 -> "Computer Engineering"
```

**Arguments for normalizing it into its own table:** consistent spelling (no `"Computer Eng."` vs. `"Computer Engineering"` typos), easy to update a major's name in one place, easy to run `GROUP BY` reports on enrollment per major.

**Arguments for keeping it as free text (as done here):** fewer joins for a simple student-registration form, less upfront design overhead if the list of majors isn't well-defined yet.

**Practical recommendation:** `FieldOfStudy` is a strong candidate to normalize into its own `Major` table in a real system (few, well-defined values, frequently filtered/grouped on) -- exactly the same reasoning as `ProductCategory` in Session 7. `Education` and `Job`, by contrast, tend to have far more varied, free-form values in practice, so plain text is often the more practical choice unless the institution enforces a fixed list.

## 10. Three Ways to Model `Education` (in Increasing Order of Normalization)

`Education` deserves its own decision tree, because how much detail you actually need changes the right design entirely.

### Option 1 -- Just a Level (simplest)

If all you need is one value like `High School`, `Bachelor`, `Master`, `PhD`:

```sql
EducationLevel NVARCHAR(20)
```

Good enough when the level itself is the only fact that matters.

### Option 2 -- Level + Field + University + Year (still one row, more detail)

Cramming everything into one free-text column is a mistake:

```text
[BAD]  Education = "MSc Applied Mathematics, Shahid Beheshti University"
```

This is a 1NF violation in disguise -- it's not an atomic value, it's three facts glued together as text, impossible to filter or sort on individually. Split it into real columns instead:

```sql
EducationLevel   NVARCHAR(20)
FieldOfStudy     NVARCHAR(100)
University       NVARCHAR(100)
GraduationYear   INT
```

### Option 3 -- A Separate `Education` Table (most correct, when a person can have multiple degrees)

If a person can hold **more than one** degree -- a Bachelor's, then a Master's, then a PhD -- `Education` shouldn't be a column on `Person` at all. This is exactly the "can there be more than one of this per entity instance?" question from Session 9:

```text
Person
    |
    +-- Bachelor
    +-- Master
    +-- PhD
```

```sql
CREATE TABLE Education (
    EducationID     INT IDENTITY(1,1) PRIMARY KEY,
    PersonID        INT NOT NULL REFERENCES Person(PersonID),
    Degree          NVARCHAR(20)  NOT NULL,
    FieldOfStudy    NVARCHAR(100) NOT NULL,
    University      NVARCHAR(100) NOT NULL,
    StartYear       INT NULL,
    GraduationYear  INT NULL
);
```

This is a `Person 1 --- * Education` relationship -- one more instance of the Session 6-7 pattern (`Orders 1 --- * OrderDetails`), just applied to academic history instead of purchases.

### Which One Should You Actually Pick?

| Question | Answer -> Option |
|---|---|
| Only the highest/current level matters | Option 1 (`EducationLevel` alone) |
| One degree's full detail matters | Option 2 (four separate columns) |
| A person can have multiple degrees over time | Option 3 (separate `Education` table) |

For a practice exercise, start with whichever matches what you're actually trying to record -- but Option 3 is the version that reflects real academic history correctly.

## 11. A Second Worked Example: `Instructor` -- Applying the Same Process to a New Entity

Same seven-step process as the `Student` table, applied to a different entity, to reinforce that this is a repeatable procedure, not something specific to students.

| Attribute | Column | Data Type | Nullable? | Constraint | Reasoning |
|---|---|---|---|---|---|
| -- | `InstructorID` | `INT IDENTITY(1,1)` | NOT NULL | `PRIMARY KEY` | Surrogate Key |
| کد بیمه | `InsuranceCode` | `VARCHAR(20)` | NOT NULL | `UNIQUE` | An identifier, not a quantity (see Section 4b) -- never `INT` |
| تاریخ شروع همکاری | `StartDate` | `DATE` | NOT NULL | `CHECK (StartDate <= GETDATE())` | Only the date matters, no time-of-day needed |

### Why `InternationalCertificates` Is a Separate Table, Not a Column

The same instinct from `Education`'s Option 3 applies here even more clearly -- one instructor can easily hold several certificates (IELTS, PMP, AWS), so a single `NVARCHAR(500)` column trying to hold all of them as text is exactly the kind of free-text-cramming Option 2 above warned against:

```text
[BAD]  InternationalCertificates NVARCHAR(500) = "IELTS, PMP, AWS Certified"
```

```sql
CREATE TABLE InternationalCertificate (
    CertificateID       INT IDENTITY(1,1) PRIMARY KEY,
    InstructorID         INT NOT NULL REFERENCES Instructor(InstructorID),
    CertificateName      NVARCHAR(100) NOT NULL,
    IssuingOrganization  NVARCHAR(100) NULL,
    IssueDate            DATE NULL,
    ExpiryDate           DATE NULL,
    CertificateNumber    VARCHAR(50) NULL
);
```

`Instructor 1 --- * InternationalCertificate` -- the exact same One-to-Many pattern as `Person 1 --- * Education` above, and as `Orders 1 --- * OrderDetails` back in Session 6.

## 12. Writing Good Column Descriptions (Data Dictionary Practice)

Recall the **Data Dictionary** concept -- every column deserves a one-line description explaining what it holds. A few worked examples, in the style a real schema's documentation would use:

| Column | Description |
|---|---|
| `PersianFirstName NVARCHAR(20)` | The person's first name written in Persian characters. |
| `EnglishFirstName NVARCHAR(20)` | The person's first name written using English characters. |
| `StudentCode INT` | A unique identification code assigned to each student. |
| `NationalCode CHAR(10)` | The person's 10-digit national identification code. |
| `InsuranceCode VARCHAR(20)` | The instructor's social security insurance number. |
| `StartDate DATE` | The date when the instructor started working at the institution. |

**A good description does two things:** states plainly what the column represents, and -- where it isn't obvious from the name alone -- hints at *why* the data type was chosen (e.g., noting a code is fixed-length, or that a date excludes time). This habit turns the schema itself into documentation that survives long after the original design conversation is forgotten.